# Chapter 3 — Data Visualization: Network Traffic Dashboard

## 📋 Latihan Soal

### Dataset
- **Mirai Botnet Dataset** — https://huggingface.co/datasets/bornpresident/mirai_botnet
- 1.083.206 baris, 48 kolom
- Sample 5000 baris untuk visualisasi
- Library: matplotlib, seaborn, plotnine (ggplot), bokeh


In [ ]:
# ============================================
# SETUP: Load Mirai Botnet Dataset
# ============================================
# Dataset: https://huggingface.co/datasets/bornpresident/mirai_botnet
# 1.083.206 baris, 48 kolom, 4 kelas traffic
# Label: BenignTraffic, Mirai-greeth_flood, Mirai-udpplain, Mirai-greip_flood
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load dataset dari HuggingFace
ds = load_dataset('bornpresident/mirai_botnet')
df = ds['train'].to_pandas()

# Rename kolom agar lebih mudah dipakai (ganti spasi dengan underscore)
df.columns = [c.replace(' ', '_') for c in df.columns]

# Rename kolom 'label' jadi 'traffic_type' agar konsisten
df = df.rename(columns={'label': 'traffic_type'})

# Konversi is_benign ke format Benign/Malicious
df['traffic_type'] = df['traffic_type'].apply(
    lambda x: 'Benign' if x == 'BenignTraffic' else 'Malicious'
)

# Ambil sample 5000 baris untuk performa notebook
# (dataset penuh 1M+ baris — sample cukup untuk EDA)
np.random.seed(42)
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Traffic types:\n{df['traffic_type'].value_counts()}")
print(f"\nKolom ({len(df.columns)} total):")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} ({df[col].dtype})")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")


---
## Soal 1: Data Profiling untuk Visualisasi

Profil data sebelum visualisasi: cek shape, distribusi, missing values, dan statistik deskriptif.


In [ ]:

# Data profiling
print(f"Shape: {df.shape}")
print(f"Kolom: {len(df.columns)}")
print(f"\nDistribusi traffic type:")
print(df['traffic_type'].value_counts())

print(f"\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada missing values")

print(f"\nStatistik deskriptif (kolom numerik):")
print(df.describe().round(2))


---
## Soal 2: Matplotlib — Time Series & Distribusi

Buat 3 visualisasi:
1. **Line chart**: Packet Rate sample (500 data points)
2. **Histogram**: Distribusi Rate
3. **Scatter plot**: Rate vs Tot_size, color by traffic_type


In [ ]:

import matplotlib.pyplot as plt

# 1. Time Series - Rate over time
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(df['Rate'].values[:500], alpha=0.7, linewidth=0.5)
axes[0].set_title('Packet Rate (sample 500)')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Rate (pkts/s)')

# 2. Histogram - Rate distribution
axes[1].hist(df['Rate'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribusi Packet Rate')
axes[1].set_xlabel('Rate')
axes[1].set_ylabel('Frekuensi')

# 3. Scatter - Rate vs Tot_size
colors = df['traffic_type'].map({'Benign': 'green', 'Malicious': 'red'})
axes[2].scatter(df['Rate'].values, df['Tot_size'].values, c=colors, alpha=0.3, s=10)
axes[2].set_title('Rate vs Total Size')
axes[2].set_xlabel('Rate')
axes[2].set_ylabel('Tot_size')

plt.tight_layout()
plt.savefig('/tmp/ch3_matplotlib.png', dpi=150)
print("✅ Matplotlib charts saved to /tmp/ch3_matplotlib.png")


---
## Soal 3: Seaborn — Heatmap & Boxplot

Buat 2 visualisasi:
1. **Heatmap**: Korelasi antar fitur numerik utama
2. **Boxplot**: Distribusi Rate per Protocol Type


In [ ]:

import seaborn as sns

# 1. Correlation Heatmap
plt.figure(figsize=(12, 8))
corr_cols = ['flow_duration', 'Rate', 'Duration', 'syn_flag_number', 'Tot_size', 'AVG', 'Std', 'Weight']
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Heatmap Korelasi Fitur Numerik')
plt.tight_layout()
plt.savefig('/tmp/ch3_heatmap.png', dpi=150)
print("✅ Heatmap saved to /tmp/ch3_heatmap.png")

# 2. Boxplot - Rate per Protocol Type
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Protocol_Type', y='Rate')
plt.title('Distribusi Rate per Protocol Type')
plt.tight_layout()
plt.savefig('/tmp/ch3_boxplot.png', dpi=150)
print("✅ Boxplot saved to /tmp/ch3_boxplot.png")


---
## Soal 4: Seaborn — Violin Plot & Protocol Pattern

Buat 2 visualisasi:
1. **Violin plot**: Distribusi Rate per Traffic Type
2. **Bar chart**: Rata-rata Rate & Tot_size per Protocol Type


In [ ]:

# 1. Violin Plot
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x='traffic_type', y='Rate', inner='quartile')
plt.title('Distribusi Rate per Traffic Type')
plt.tight_layout()
plt.savefig('/tmp/ch3_violin.png', dpi=150)
print("✅ Violin plot saved to /tmp/ch3_violin.png")

# 2. Pola per Protocol
plt.figure(figsize=(8, 5))
proto_avg = df.groupby('Protocol_Type')[['Rate', 'Tot_size']].mean()
proto_avg.plot(kind='bar', ax=plt.gca())
plt.title('Rata-rata Rate & Tot_size per Protocol Type')
plt.tight_layout()
plt.savefig('/tmp/ch3_protocol_pattern.png', dpi=150)
print("✅ Protocol pattern saved to /tmp/ch3_protocol_pattern.png")


---
## Soal 5: GGPLOT Style (plotnine)

Buat scatter plot Rate vs Tot_size menggunakan **plotnine** (ggplot2 untuk Python). Facet per traffic_type.


In [ ]:

# Install: pip install plotnine
try:
    from plotnine import ggplot, aes, geom_point, facet_wrap, labs
    
    p = (ggplot(df, aes(x='Rate', y='Tot_size', color='traffic_type'))
         + geom_point(alpha=0.5, size=2)
         + facet_wrap('~traffic_type')
         + labs(title='Rate vs Tot_size per Traffic Type', x='Rate', y='Tot_size'))
    p.save('/tmp/ch3_ggplot.png', dpi=150, width=10, height=5)
    print("✅ plotnine chart saved")
except ImportError:
    # Fallback ke matplotlib
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, tt in enumerate(df['traffic_type'].unique()):
        sub = df[df['traffic_type'] == tt]
        axes[i].scatter(sub['Rate'].values, sub['Tot_size'].values, alpha=0.3, s=10)
        axes[i].set_title(f'Traffic Type: {tt}')
    plt.tight_layout()
    plt.savefig('/tmp/ch3_ggplot_fallback.png', dpi=150)
    print("✅ Fallback matplotlib chart saved")


---
## Soal 6: Bokeh — Interactive Visualization

Buat scatter plot interaktif dengan **Bokeh**:
1. Hover tooltip menampilkan detail setiap titik
2. Box select tool untuk zoom


In [ ]:

# Install: pip install bokeh
try:
    from bokeh.plotting import figure, output_file, save
    from bokeh.models import HoverTool
    
    output_file('/tmp/ch3_bokeh.html')
    
    p = figure(title='Rate vs Tot_size (interactive)', width=700, height=400)
    p.circle(df['Rate'].values[:1000], df['Tot_size'].values[:1000],
             alpha=0.5, size=6, color='navy')
    p.add_tools(HoverTool(tooltips=[('Rate', '@x'), ('Tot_size', '@y')]))
    p.xaxis.axis_label = 'Rate'
    p.yaxis.axis_label = 'Tot_size'
    save(p)
    print("✅ Bokeh interactive chart saved to /tmp/ch3_bokeh.html")
except ImportError:
    print("⚠️ bokeh tidak terinstall — lewati")


---
## Soal 7: Multi-Panel Dashboard

Buat dashboard 6 panel yang menggabungkan semua insight dari visualisasi sebelumnya.


In [ ]:

# Dashboard 6 panel
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Rate distribution
axes[0,0].hist(df['Rate'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0,0].set_title('Distribusi Rate')

# Panel 2: Traffic type count
df['traffic_type'].value_counts().plot(kind='bar', ax=axes[0,1], color=['green', 'red'])
axes[0,1].set_title('Jumlah per Traffic Type')
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=30)

# Panel 3: Protocol distribution
df['Protocol_Type'].value_counts().plot(kind='bar', ax=axes[0,2], color='coral')
axes[0,2].set_title('Distribusi Protocol Type')

# Panel 4: Heatmap
corr_cols = ['flow_duration', 'Rate', 'Duration', 'syn_flag_number', 'Tot_size', 'AVG']
sns.heatmap(df[corr_cols].corr(), ax=axes[1,0], annot=True, fmt='.2f', cmap='coolwarm', center=0)
axes[1,0].set_title('Heatmap Korelasi')

# Panel 5: Boxplot
sns.boxplot(data=df, x='traffic_type', y='Rate', ax=axes[1,1])
axes[1,1].set_title('Boxplot Rate per Traffic Type')

# Panel 6: Rate by protocol
df.groupby('Protocol_Type')['Rate'].mean().plot(kind='bar', ax=axes[1,2], color='teal')
axes[1,2].set_title('Rata-rata Rate per Protocol')

plt.suptitle('Mirai Botnet Traffic Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/ch3_dashboard.png', dpi=150, bbox_inches='tight')
print("✅ Dashboard 6-panel saved to /tmp/ch3_dashboard.png")
